In [0]:
print("PySpark is working!")

df = spark.range(10)

display(df)

In [0]:
print("PySpark is working!")

df = spark.range(1, 1000000)

print("Rows:", df.count())

display(df.limit(10))

In [0]:
from pyspark.sql import functions as F

# Generate 1 million transactions
transactions = (
    spark.range(1, 1_000_001)
    .withColumnRenamed("id", "transaction_id")
    .withColumn(
        "customer_id",
        (F.rand(seed=42) * 100_000).cast("int")
    )
    .withColumn(
        "product_id",
        (F.rand(seed=43) * 10_000).cast("int")
    )
    .withColumn(
        "quantity",
        (F.rand(seed=44) * 5 + 1).cast("int")
    )
    .withColumn(
        "unit_price",
        F.round(F.rand(seed=45) * 990 + 10, 2)
    )
    .withColumn(
        "transaction_date",
        F.date_add(
            F.lit("2025-01-01"),
            (F.rand(seed=46) * 365).cast("int")
        )
    )
)

transactions.show(10)

In [0]:
print("Rows:", transactions.count())
transactions.printSchema()
display(transactions.limit(20))

In [0]:

from pyspark.sql import functions as F

customers = (
    spark.range(1, 100_001)
    .withColumnRenamed("id", "customer_id")
    .withColumn(
        "customer_name",
        F.concat(F.lit("Customer_"), F.col("customer_id"))
    )
    .withColumn(
        "city",
        F.element_at(
            F.array(
                F.lit("Pune"),
                F.lit("Mumbai"),
                F.lit("Delhi"),
                F.lit("Bangalore"),
                F.lit("Hyderabad")
            ),
            (F.rand(seed=101) * 5 + 1).cast("int")
        )
    )
    .withColumn(
        "customer_segment",
        F.element_at(
            F.array(
                F.lit("Premium"),
                F.lit("Standard"),
                F.lit("Basic")
            ),
            (F.rand(seed=102) * 3 + 1).cast("int")
        )
    )
)

print("Customers:", customers.count())
display(customers.limit(10))

In [0]:
products = (
    spark.range(1, 10_001)
    .withColumnRenamed("id", "product_id")
    .withColumn(
        "product_name",
        F.concat(F.lit("Product_"), F.col("product_id"))
    )
    .withColumn(
        "category",
        F.element_at(
            F.array(
                F.lit("Electronics"),
                F.lit("Clothing"),
                F.lit("Home"),
                F.lit("Grocery"),
                F.lit("Sports")
            ),
            (F.rand(seed=201) * 5 + 1).cast("int")
        )
    )
)

print("Products:", products.count())
display(products.limit(10))

In [0]:
c = customers.alias("c")
t = transactions.alias("t")
result = (
    t.join(c, t.customer_id == c.customer_id, "inner")
     .select(
         t.transaction_id,
         t.customer_id,
         c.customer_name,
         c.city,
         c.customer_segment,
         t.product_id,
         t.quantity,
         t.unit_price,
         t.transaction_date
     )
)
display(result)

In [0]:
print("Transactions:", transactions.count())
print("Joined:", result.count())
transactions.filter(F.col("customer_id") == 0).count()

In [0]:
c = customers.alias("c")
t = transactions.alias("t")
valid_transactions = c.join(t, c.customer_id == t.customer_id, "left").count()
print("Valid Transactions:", valid_transactions)

In [0]:
c = customers.alias("c")
t = transactions.alias("t")
invalid_transactions = c.join(t, c.customer_id == t.customer_id, "leftanti").count()
print("Invalid Transactions:", invalid_transactions)

In [0]:
print("Session check")
print(spark.version)

In [0]:
print("transactions" in dir())
print("customers" in dir())

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.pyspark_deep_dive;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pyspark_deep_dive.transactions")

In [0]:
transactions = spark.table(
    "workspace.pyspark_deep_dive.transactions"
)

In [0]:
customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pyspark_deep_dive.customers")

In [0]:
products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pyspark_deep_dive.products")

In [0]:
spark.sql("""
SHOW TABLES IN workspace.pyspark_deep_dive
""").show()

In [0]:
transactions = spark.table("workspace.pyspark_deep_dive.transactions")
customers = spark.table("workspace.pyspark_deep_dive.customers")
products = spark.table("workspace.pyspark_deep_dive.products")

print("Transactions:", transactions.count())
print("Customers:", customers.count())
print("Products:", products.count())